# Notebook 05d — Where Is the Evidence?

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 05d of 12

---

### What notebook 05c found

The corrected control is sound — the example figure confirms an identical square on intact and
cracked eggs alike — and the result was consistent across every fold.

| condition | four-class | binary |
|---|---|---|
| clean | 0.7431 | 0.8717 |
| crack occluded, 65 % covered | 0.7121 | 0.8391 |
| **crack occluded, 95 % covered** | **0.6578** | **0.8330** |
| acquisition floor (no pixels) | 0.295 | 0.565 |

The occlusion hurts, consistently: the paired drop is 0.0853 ± 0.0334 with the same sign on all
five folds. But it removes only **19 % of the four-class skill and 13 % of the binary skill**
measured above the acquisition floor. With 95 % of annotated cracks painted out, the model still
calls **86.7 %** of those eggs cracked.

So the model's decision is **not localised to the annotated crack region**. That is a real finding
and it needs explaining rather than burying. Three explanations are possible and they have very
different consequences:

1. **Crack evidence outside the polygon.** The annotations mark the crack a human drew round, not
   every pixel it affects. Hairline cracks have fine extensions; Network crazing is diffuse across
   much of the shell and the polygon marks a representative patch, not the whole field.
2. **Genuine whole-shell signal.** A cracked egg may differ globally — translucency, mottling,
   shell thinning — which is real biology and legitimately diagnostic, just not "the crack".
3. **Something unwanted** that survived the preprocessing and augmentation work.

Explanation 3 is largely ruled out already: the occluded model scores 0.658 against an acquisition
floor of 0.295, so it is not living on capture artefacts. Distinguishing 1 from 2 is what this
notebook does.

### The experiments

| | Experiment | Question |
|---|---|---|
| A | **Inverse control** — keep only the crack square, blank the other 58 % of the frame | Is the crack **sufficient**? |
| B | Larger occlusion, 0.75 (98.8 % coverage) | Does the trend continue towards the floor as coverage approaches complete? |
| C | Per-pattern breakdown of every condition | Is the residual driven by Network, whose crazing a single box cannot remove? |

Pairing A with 05c's result gives the full picture. Occluding the crack asks whether it is
*necessary*; keeping only the crack asks whether it is *sufficient*. Both can be true at once —
that is what redundant evidence looks like, and it is a perfectly respectable finding.

### Why this matters beyond Chapter 5

Objective 3 claims explainable visual evidence, and notebook 09 measures whether Grad-CAM lands on
the annotation polygons. If the model is genuinely using whole-shell appearance, **those heatmaps
will not concentrate on the polygons, and that will not be a failure of the explainability method**
— it will be the explanation correctly reporting what the model does. Establishing this now means
notebook 09's numbers get interpreted properly instead of being read as a broken Grad-CAM.

---
## 0. Configuration

In [ ]:
import os
os.environ["EGG_ARTEFACT_ROOT"] = r"E:\Degree\Final Research\Egg_Crack_Dataset\Eggshell-Diagnostic-Research\Artefact"

BACKBONE    = "efficientnet_b0"
FORMULATION = "single4"
N_FOLDS     = 5

RUN_INVERSE   = False     # experiment A: keep only the crack
RUN_BIG_BOX   = False     # experiment B: occlude 0.75
INVERSE_FRAC  = 0.65      # same square as the 95%-coverage condition, so the two are comparable
BIG_BOX_FRAC  = 0.75

CACHE_VARIANT = "neutral_fill"; CACHE_SIDE = 448; TRAIN_SIDE = 384
POLICY = "medium"; EPOCHS = 45; PATIENCE = 10
EFFECTIVE_BATCH = 32; PHYS_BATCH = 32; NUM_WORKERS = 4

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime

_here = Path.cwd()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _src is None:
    raise RuntimeError("Run from Artefact/notebooks/ or set EGG_ARTEFACT_ROOT.")
sys.path.insert(0, str(_src))

import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import f1_score, recall_score

from common import project_config as CFG
from common.data import EggDataset, build_crack_masks, make_loader, class_weights
from common.engine import TrainConfig, train_fold, predict, append_run
from common.models import build_model

warnings.filterwarnings("ignore")
CFG.seed_everything()

STAGE = "05d_evidence_localisation"
OUT = CFG.results_dir(STAGE); FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PROC = CFG.DATA_DIR / "processed"; RUNS_CSV = CFG.RESULTS_DIR / "runs.csv"
PRED_DIR = OUT / "predictions"; PRED_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

splits = pd.read_csv(PROC / "splits.csv")
splits["label4"] = pd.Categorical(splits["label4"], categories=CFG.LABEL4, ordered=True)
CACHE_PATH = PROC / f"cache_{CACHE_VARIANT}_{CACHE_SIDE}.npy"
y4 = splits["label4"].map(CFG.LABEL4_TO_IDX).to_numpy(dtype=np.int64)
y2 = splits["is_cracked"].to_numpy(dtype=np.int64)
ypat = np.where(y4 > 0, y4 - 1, -100).astype(np.int64)
age = splits["age_class"].map(CFG.AGE_TO_IDX).to_numpy(dtype=np.int64)
LABELS = {"y4": y4, "y2": y2, "ypat": ypat, "age": age}
MASKS = build_crack_masks(splits["crack_region"].tolist(), TRAIN_SIDE)

REF = {"clean_4": 0.7431, "clean_2": 0.8717, "floor_4": 0.295, "floor_2": 0.565,
       "occ45_4": 0.7121, "occ45_2": 0.8391, "occ65_4": 0.6578, "occ65_2": 0.8330}
S = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds"),
     "reference": REF}
print(f"device {device} | reference clean {REF['clean_4']} / occluded0.65 {REF['occ65_4']}")

In [ ]:
def fold_rows(k):
    tr = splits.index[(splits["split"].eq("dev")) & (splits["fold"] != k)].to_numpy()
    va = splits.index[(splits["split"].eq("dev")) & (splits["fold"] == k)].to_numpy()
    return tr, va

def already(tag, fold):
    if not RUNS_CSV.exists():
        return False
    try:
        d = pd.read_csv(RUNS_CSV)
    except Exception:
        return False
    return bool(len(d) and "tag" in d.columns and ((d["tag"] == tag) & (d["fold"] == fold)).any())

def run_condition(tag, frac, invert):
    for k in range(N_FOLDS):
        if already(tag, k):
            print(f"  fold {k}: already recorded"); continue
        tr, va = fold_rows(k)
        cfg = TrainConfig(backbone=BACKBONE, formulation=FORMULATION, n_out=4, policy=POLICY,
                          train_side=TRAIN_SIDE, epochs=EPOCHS, physical_batch=PHYS_BATCH,
                          effective_batch=EFFECTIVE_BATCH, patience=PATIENCE,
                          num_workers=NUM_WORKERS, seed=CFG.RANDOM_SEED + k,
                          occlude_crack=True, tag=tag)
        kw = dict(masks=MASKS, occlude_crack=True, occlude_box_frac=frac,
                  occlude_invert=invert)
        ds_tr = EggDataset(CACHE_PATH, tr, LABELS, TRAIN_SIDE, policy=POLICY, **kw)
        ds_va = EggDataset(CACHE_PATH, va, LABELS, TRAIN_SIDE, policy=None, **kw)
        ltr = make_loader(ds_tr, PHYS_BATCH, True, NUM_WORKERS)
        lva = make_loader(ds_va, PHYS_BATCH, False, NUM_WORKERS)
        m, _, state = train_fold(cfg, ltr, lva, device, class_weights(y4[tr], 4), None, None,
                                 verbose=False)
        append_run(RUNS_CSV, cfg, k, m,
                   {"n_train": len(tr), "n_val": len(va), "box_frac": frac,
                    "invert": int(invert)})
        mdl = build_model(BACKBONE, FORMULATION, 4, pretrained=False, img_size=TRAIN_SIDE)
        mdl.load_state_dict(state); mdl.to(device).eval()
        p = predict(mdl, lva, device, cfg)
        np.savez(PRED_DIR / f"{tag}_fold{k}.npz", idx=p["idx"], y4=p["y4"],
                 pred4=p["logits"].argmax(1))
        del mdl; torch.cuda.empty_cache()
        print(f"  fold {k}: macroF1 {m['macro_f1']:.4f}  "
              f"bin {m.get('macro_f1_binary', float('nan')):.4f}")

---
## 1. What the two conditions look like

Before training, confirm visually that the inverse condition really does keep the crack and discard
everything else. If the surviving square does not contain the crack, the experiment answers a
different question from the one intended.

In [ ]:
cache = np.load(CACHE_PATH, mmap_mode="r")
picks = [splits.index[splits["label4"].eq(c) & splits["split"].eq("dev")][
             len(splits.index[splits["label4"].eq(c) & splits["split"].eq("dev")]) // 2]
         for c in CFG.LABEL4]
conds = [("clean", dict()),
         (f"occluded {INVERSE_FRAC}", dict(occlude_crack=True, occlude_box_frac=INVERSE_FRAC)),
         (f"inverse {INVERSE_FRAC}", dict(occlude_crack=True, occlude_box_frac=INVERSE_FRAC,
                                          occlude_invert=True))]
fig, axes = plt.subplots(len(conds), len(picks), figsize=(3.1 * len(picks), 3.3 * len(conds)))
for j, r in enumerate(picks):
    for i, (lab, kw) in enumerate(conds):
        ds = EggDataset(CACHE_PATH, [r], LABELS, TRAIN_SIDE, policy=None, masks=MASKS, **kw)
        x = ds[0]["image"].numpy().transpose(1, 2, 0)
        x = (x - x.min()) / (x.max() - x.min() + 1e-9)
        axes[i, j].imshow(x)
        axes[i, j].set_title((f"{splits.loc[r, 'egg_id']} — {splits.loc[r, 'label4']}\n{lab}")
                             if i == 0 else lab, fontsize=8)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([]); axes[i, j].grid(False)
fig.suptitle("Occluding the crack (middle) versus keeping only the crack (bottom)",
             y=1.0, fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_conditions.png", bbox_inches="tight")
plt.show()
print("In the bottom row the visible square should contain the crack for the three cracked eggs.")

---
## 2. Run

In [ ]:
if RUN_INVERSE:
    print(f"\nEXPERIMENT A -- inverse control, keep only the {INVERSE_FRAC} square")
    run_condition(f"inverse_box{int(INVERSE_FRAC * 100)}", INVERSE_FRAC, True)
if RUN_BIG_BOX:
    print(f"\nEXPERIMENT B -- larger occlusion, {BIG_BOX_FRAC}")
    run_condition(f"control_box{int(BIG_BOX_FRAC * 100)}", BIG_BOX_FRAC, False)
if not (RUN_INVERSE or RUN_BIG_BOX):
    print("nothing scheduled; set RUN_INVERSE and RUN_BIG_BOX")

---
## 3. Results

In [ ]:
runs = pd.read_csv(RUNS_CSV) if RUNS_CSV.exists() else pd.DataFrame()

def summarise(tag):
    r = runs[runs["tag"].eq(tag)] if len(runs) and "tag" in runs.columns else pd.DataFrame()
    if not len(r):
        return None
    return {"four_class": round(r["macro_f1"].mean(), 4),
            "sd": round(r["macro_f1"].std(), 4),
            "binary": round(r["macro_f1_binary"].mean(), 4),
            "folds": int(r["fold"].nunique())}

rows = [{"condition": "clean (whole egg)", "four_class": REF["clean_4"], "sd": 0.0316,
         "binary": REF["clean_2"], "frame_kept": "100%"},
        {"condition": "crack occluded 0.45", "four_class": REF["occ45_4"], "sd": 0.0517,
         "binary": REF["occ45_2"], "frame_kept": "80%"},
        {"condition": "crack occluded 0.65", "four_class": REF["occ65_4"], "sd": 0.0463,
         "binary": REF["occ65_2"], "frame_kept": "58%"}]
for tag, name, kept in [(f"control_box{int(BIG_BOX_FRAC * 100)}",
                         f"crack occluded {BIG_BOX_FRAC}", "44%"),
                        (f"inverse_box{int(INVERSE_FRAC * 100)}",
                         f"ONLY the crack kept ({INVERSE_FRAC})", "42%")]:
    v = summarise(tag)
    if v:
        rows.append({"condition": name, "four_class": v["four_class"], "sd": v["sd"],
                     "binary": v["binary"], "frame_kept": kept})
rows.append({"condition": "acquisition floor (no pixels)", "four_class": REF["floor_4"],
             "sd": np.nan, "binary": REF["floor_2"], "frame_kept": "0%"})
res = pd.DataFrame(rows).set_index("condition")
res.to_csv(TAB_DIR / "t01_evidence_conditions.csv")
S["conditions"] = res.reset_index().to_dict("records")
print(res.to_string())

inv = summarise(f"inverse_box{int(INVERSE_FRAC * 100)}")
if inv:
    span4 = REF["clean_4"] - REF["floor_4"]
    suff = (inv["four_class"] - REF["floor_4"]) / span4
    nec = (REF["clean_4"] - REF["occ65_4"]) / span4
    S["sufficiency"] = round(float(suff), 3); S["necessity"] = round(float(nec), 3)
    print(f"\nNECESSITY  : removing the crack costs {nec:.0%} of the skill above the floor")
    print(f"SUFFICIENCY: the crack alone retains {suff:.0%} of the skill above the floor")
    print()
    if suff > 0.6 and nec < 0.35:
        print("The crack is SUFFICIENT but not NECESSARY. The evidence is redundant: the crack")
        print("carries the signal, and so does the rest of the shell. This is the honest finding,")
        print("and it predicts that Grad-CAM in notebook 09 will be diffuse rather than focused")
        print("on the polygons -- which will be the explanation working, not failing.")
    elif suff > 0.6 and nec >= 0.35:
        print("The crack is both sufficient and substantially necessary: the model is genuinely")
        print("crack-driven and the explainability claim in Objective 3 is on firm ground.")
    elif suff <= 0.6 and nec < 0.35:
        print("The crack is neither sufficient nor necessary on its own. The model integrates")
        print("whole-shell appearance. Objective 3's claims must be reworded around shell-level")
        print("evidence rather than crack localisation.")
    else:
        print("Mixed result: report both numbers and let the Grad-CAM analysis in notebook 09")
        print("arbitrate.")

In [ ]:
# --- per-pattern breakdown: is Network driving the residual? --------------
def per_pattern(tag):
    files = sorted(PRED_DIR.glob(f"{tag}_fold*.npz"))
    if not files:
        return None
    yt, yp = [], []
    for f in files:
        z = np.load(f); yt.append(z["y4"]); yp.append(z["pred4"])
    yt, yp = np.concatenate(yt), np.concatenate(yp)
    out = {}
    for ci, cname in enumerate(CFG.LABEL4):
        m = yt == ci
        if m.sum():
            out[cname] = round(float(recall_score(m.astype(int), (yp == ci).astype(int),
                                                  zero_division=0)), 3)
    return out

pp_rows = []
for tag, name in [(f"control_box65", "crack occluded 0.65"),
                  (f"control_box{int(BIG_BOX_FRAC * 100)}", f"crack occluded {BIG_BOX_FRAC}"),
                  (f"inverse_box{int(INVERSE_FRAC * 100)}", "only the crack kept")]:
    v = per_pattern(tag)
    if v:
        pp_rows.append({"condition": name, **v})
if pp_rows:
    pp = pd.DataFrame(pp_rows).set_index("condition")
    pp.to_csv(TAB_DIR / "t02_per_pattern_recall.csv")
    S["per_pattern"] = pp.reset_index().to_dict("records")
    print("per-class recall under each condition:")
    print(pp.to_string())
    print("\nNetwork is diffuse crazing spread over much of the shell, so a single square cannot")
    print("remove it. If Network recall survives occlusion while Hairline and Star fall, the")
    print("residual skill is explained by incomplete removal rather than by context reading.")

In [ ]:
if len(res) > 3:
    fig, ax = plt.subplots(figsize=(9.2, 4.6))
    names = list(res.index); vals = res["four_class"].to_numpy()
    cols = []
    for n in names:
        cols.append("#4C72B0" if n.startswith("clean") else
                    "#55A868" if "ONLY" in n else
                    "#C44E52" if "floor" in n else "#DD8452")
    ax.barh(names, vals, color=cols, alpha=0.9,
            xerr=res["sd"].fillna(0).to_numpy(), capsize=3)
    for i, (v, k) in enumerate(zip(vals, res["frame_kept"])):
        ax.text(v + 0.012, i, f"{v:.3f}   ({k} of frame)", va="center", fontsize=8)
    ax.axvline(REF["floor_4"], ls="--", color="#C44E52", lw=1.2)
    ax.axvline(REF["clean_4"], ls="--", color="#4C72B0", lw=1.2)
    ax.invert_yaxis(); ax.set_xlim(0, 1.05)
    ax.set_xlabel("four-class macro F1")
    ax.set_title("Where does the evidence live?")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "02_evidence_summary.png", bbox_inches="tight")
    plt.show()

def _safe(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)
(OUT / "evidence_summary.json").write_text(json.dumps(S, indent=2, default=_safe))

L = ["=" * 70, "NOTEBOOK 05d DIGEST -- EVIDENCE LOCALISATION", "=" * 70,
     f"generated: {S['generated_at']}", ""]
for r in S.get("conditions", []):
    L.append(f"   {r['condition']:<36s} 4cls {r['four_class']}  bin {r['binary']}  "
             f"frame kept {r['frame_kept']}")
if "sufficiency" in S:
    L += ["", f"NECESSITY  (skill lost when the crack is removed): {S['necessity']:.0%}",
          f"SUFFICIENCY (skill retained from the crack alone)  : {S['sufficiency']:.0%}"]
if "per_pattern" in S:
    L += ["", "PER-CLASS RECALL"]
    for r in S["per_pattern"]:
        L.append("   " + "  ".join(f"{k}={v}" for k, v in r.items()))
L.append("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print(digest)

---
## 5. Correction to the session-grouped evaluation

The 0.006 generalisation gap reported in notebook 05b should not go into the dissertation as it
stands. That evaluation held out capture blocks 0 to 4, and blocks 0, 1 and 2 contain **no Star
eggs at all**:

| block | Intact | Hairline | Star | Network |
|---|---|---|---|---|
| 0 | 59 | 0 | **0** | 6 |
| 1 | 51 | 9 | **0** | 5 |
| 2 | 48 | 9 | **0** | 8 |
| 3 | 36 | 14 | 6 | 9 |
| 4 | 18 | 28 | 10 | 9 |

`f1_score(average="macro")` averages over whichever labels appear in the true or predicted set, so
the denominator changed from block to block — three classes in some, four in others. Block 0 scored
0.626 and block 1 scored 0.890, and part of that difference is an artefact of which classes
existed rather than of generalisation.

Two fixes: macro F1 over a **fixed four-label set**, and only blocks that contain **all four
classes**. The excluded blocks are named rather than quietly dropped.

In [ ]:
from sklearn.metrics import f1_score, recall_score

ALL4 = [0, 1, 2, 3]
def macro4(yt, yp):
    """Macro F1 over a FIXED four-label set, so the denominator never changes between blocks."""
    return float(f1_score(yt, yp, labels=ALL4, average="macro", zero_division=0))

dev_mask = splits["split"].eq("dev")
blk = pd.crosstab(splits.loc[dev_mask, "capture_block"], splits.loc[dev_mask, "label4"])
usable = [int(b) for b in blk.index if (blk.loc[b] > 0).all()]
excluded = [int(b) for b in blk.index if int(b) not in usable]
print("class counts per capture block (development set):")
print(blk.to_string())
print(f"\nblocks with all four classes: {usable}")
print(f"excluded (no Star eggs): {excluded}")

In [ ]:
RUN_GROUPED_FIX = False      # set True to run; roughly 5 minutes per block

def run_grouped(tag, blocks):
    for k in blocks:
        if already(tag, int(k)):
            print(f"  block {k}: already recorded"); continue
        tr = splits.index[dev_mask & (splits["capture_block"] != k)].to_numpy()
        va = splits.index[dev_mask & (splits["capture_block"] == k)].to_numpy()
        cfg = TrainConfig(backbone=BACKBONE, formulation=FORMULATION, n_out=4, policy=POLICY,
                          train_side=TRAIN_SIDE, epochs=EPOCHS, physical_batch=PHYS_BATCH,
                          effective_batch=EFFECTIVE_BATCH, patience=PATIENCE,
                          num_workers=NUM_WORKERS, seed=CFG.RANDOM_SEED + int(k), tag=tag)
        ds_tr = EggDataset(CACHE_PATH, tr, LABELS, TRAIN_SIDE, policy=POLICY)
        ds_va = EggDataset(CACHE_PATH, va, LABELS, TRAIN_SIDE, policy=None)
        ltr = make_loader(ds_tr, PHYS_BATCH, True, NUM_WORKERS)
        lva = make_loader(ds_va, PHYS_BATCH, False, NUM_WORKERS)
        m, _, state = train_fold(cfg, ltr, lva, device, class_weights(y4[tr], 4), None, None,
                                 verbose=False)
        append_run(RUNS_CSV, cfg, int(k), m,
                   {"n_train": len(tr), "n_val": len(va), "fold_col": "capture_block"})
        mdl = build_model(BACKBONE, FORMULATION, 4, pretrained=False, img_size=TRAIN_SIDE)
        mdl.load_state_dict(state); mdl.to(device).eval()
        pr = predict(mdl, lva, device, cfg)
        np.savez(PRED_DIR / f"{tag}_fold{k}.npz", idx=pr["idx"], y4=pr["y4"], y2=pr["y2"],
                 pred4=pr["logits"].argmax(1))
        del mdl; torch.cuda.empty_cache()
        print(f"  block {k}: macroF1(fixed labels) "
              f"{macro4(pr['y4'], pr['logits'].argmax(1)):.4f}  val n={len(va)}")

if RUN_GROUPED_FIX:
    run_grouped("grouped_fixed", usable)
else:
    print("grouped correction not run")

In [ ]:
files = sorted(PRED_DIR.glob("grouped_fixed_fold*.npz"))
if files:
    z = [np.load(f) for f in files]
    yt = np.concatenate([x["y4"] for x in z])
    yp = np.concatenate([x["pred4"] for x in z])
    y2t = np.concatenate([x["y2"] for x in z]) if "y2" in z[0] else (yt > 0).astype(int)
    g4 = macro4(yt, yp)
    g2 = float(f1_score(y2t, (yp > 0).astype(int), average="macro", zero_division=0))

    per_block = []
    for f in files:
        x = np.load(f)
        per_block.append({"block": f.stem.split("fold")[-1],
                          "n": int(len(x["y4"])),
                          "macro_f1_fixed": round(macro4(x["y4"], x["pred4"]), 4)})
    pb = pd.DataFrame(per_block).set_index("block")
    pb.to_csv(TAB_DIR / "t04_grouped_per_block.csv")
    print(pb.to_string())

    cmp_tab = pd.DataFrame({
        "stratified folds": [REF["clean_4"], REF["clean_2"]],
        "session-grouped (corrected)": [round(g4, 4), round(g2, 4)],
        "classical, grouped": [0.473, 0.662],
        "acquisition floor, grouped": [0.307, 0.607],
    }, index=["four-class", "binary"])
    cmp_tab["generalisation gap"] = (cmp_tab["stratified folds"]
                                     - cmp_tab["session-grouped (corrected)"]).round(4)
    cmp_tab.to_csv(TAB_DIR / "t05_grouped_corrected.csv")
    print()
    print(cmp_tab.to_string())
    print(f"\nblocks used: {usable}; excluded for having no Star eggs: {excluded}")
    print("This supersedes the 0.006 gap in notebook 05b, which averaged over an inconsistent")
    print("label set. Report this table in Chapter 5 and the exclusion in Section 5.10.1.")
else:
    print("no corrected grouped predictions yet")